In [ ]:
import cv2
import mediapipe as mp
import numpy as np

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# =====================================================
# CONFIG
# =====================================================

IMAGE_PATH = "/Users/stella/Documents/Eng301_ComputerVision_240065/Final/CompVI-Final-Project/Photo/upload/IMG_9378.png"

MODEL_PATH = "face_landmarker.task"

# =====================================================
# LOAD MODEL
# =====================================================

base_options = python.BaseOptions(
    model_asset_path=MODEL_PATH
)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1
)

detector = vision.FaceLandmarker.create_from_options(
    options
)

# =====================================================
# HELPERS
# =====================================================

def get_point(landmarks, idx, w, h):

    lm = landmarks[idx]

    return np.array([
        int(lm.x * w),
        int(lm.y * h)
    ])

def distance(p1, p2):

    return np.linalg.norm(p1 - p2)

# =====================================================
# HEAD POSE QUALITY
# =====================================================

def estimate_pose(landmarks):

    left_cheek_x = landmarks[234].x

    right_cheek_x = landmarks[454].x

    symmetry = abs(
        left_cheek_x -
        (1 - right_cheek_x)
    )

    return symmetry

# =====================================================
# BETTER FACIAL THIRDS
# =====================================================

def analyze_facial_thirds(
    landmarks,
    w,
    h
):

    # -----------------------------------------
    # Key points
    # -----------------------------------------

    forehead_center = get_point(
        landmarks,
        10,
        w,
        h
    )

    left_brow = get_point(
        landmarks,
        105,
        w,
        h
    )

    right_brow = get_point(
        landmarks,
        334,
        w,
        h
    )

    brow_center = (
        left_brow +
        right_brow
    ) / 2

    nose_base = get_point(
        landmarks,
        2,
        w,
        h
    )

    chin = get_point(
        landmarks,
        152,
        w,
        h
    )

    # -----------------------------------------
    # Estimate hairline
    # -----------------------------------------

    forehead_to_brow = abs(
        brow_center[1] -
        forehead_center[1]
    )

    estimated_hairline_y = (
        forehead_center[1] -
        forehead_to_brow * 1.2
    )

    estimated_hairline = np.array([
        forehead_center[0],
        estimated_hairline_y
    ])

    # -----------------------------------------
    # Thirds
    # -----------------------------------------

    upper_third = abs(
        brow_center[1] -
        estimated_hairline[1]
    )

    middle_third = abs(
        nose_base[1] -
        brow_center[1]
    )

    lower_third = abs(
        chin[1] -
        nose_base[1]
    )

    total = (
        upper_third +
        middle_third +
        lower_third
    )

    upper_ratio = upper_third / total

    middle_ratio = middle_third / total

    lower_ratio = lower_third / total

    # -----------------------------------------
    # Balance score
    # -----------------------------------------

    ideal = 1 / 3

    balance_error = (
        abs(upper_ratio - ideal) +
        abs(middle_ratio - ideal) +
        abs(lower_ratio - ideal)
    )

    balance_score = max(
        0,
        1 - balance_error * 1.5
    )

    balance_score = round(
        balance_score,
        2
    )

    result = {}

    result["upper_ratio"] = round(
        upper_ratio,
        3
    )

    result["middle_ratio"] = round(
        middle_ratio,
        3
    )

    result["lower_ratio"] = round(
        lower_ratio,
        3
    )

    result["balance_score"] = (
        balance_score
    )

    # -----------------------------------------
    # Interpretation
    # -----------------------------------------

    def classify(ratio, name):

        if ratio > 0.38:
            return f"{name} hơi dài"

        elif ratio < 0.28:
            return f"{name} hơi ngắn"

        else:
            return f"{name} cân đối"

    result["upper_face"] = classify(
        upper_ratio,
        "Thượng đình"
    )

    result["middle_face"] = classify(
        middle_ratio,
        "Trung đình"
    )

    result["lower_face"] = classify(
        lower_ratio,
        "Hạ đình"
    )

    # -----------------------------------------
    # Overall
    # -----------------------------------------

    if balance_score > 0.82:

        result["overall_balance"] = (
            "Tỉ lệ gương mặt khá cân đối"
        )

    elif balance_score > 0.68:

        result["overall_balance"] = (
            "Tỉ lệ gương mặt tương đối hài hòa"
        )

    else:

        result["overall_balance"] = (
            "Tỉ lệ gương mặt hơi lệch nhẹ"
        )

    return (
        result,
        estimated_hairline.astype(int),
        brow_center.astype(int),
        nose_base.astype(int),
        chin.astype(int)
    )

# =====================================================
# ADVANCED NOSE ANALYSIS
# =====================================================

def analyze_nose(
    landmarks,
    w,
    h
):

    nose_tip = get_point(
        landmarks,
        1,
        w,
        h
    )

    left_nostril = get_point(
        landmarks,
        98,
        w,
        h
    )

    right_nostril = get_point(
        landmarks,
        327,
        w,
        h
    )

    bridge_upper = get_point(
        landmarks,
        168,
        w,
        h
    )

    left_eye = get_point(
        landmarks,
        33,
        w,
        h
    )

    right_eye = get_point(
        landmarks,
        263,
        w,
        h
    )

    forehead = get_point(
        landmarks,
        10,
        w,
        h
    )

    chin = get_point(
        landmarks,
        152,
        w,
        h
    )

    # -----------------------------------------
    # Measurements
    # -----------------------------------------

    eye_distance = distance(
        left_eye,
        right_eye
    )

    nose_width = distance(
        left_nostril,
        right_nostril
    )

    nose_length = distance(
        bridge_upper,
        nose_tip
    )

    face_height = distance(
        forehead,
        chin
    )

    width_ratio = (
        nose_width /
        eye_distance
    )

    length_ratio = (
        nose_length /
        face_height
    )

    tip_ratio = (
        nose_width /
        nose_length
    )

    # -----------------------------------------
    # 3D Bridge Projection
    # -----------------------------------------

    bridge_z = landmarks[6].z

    cheek_avg_z = (
        landmarks[234].z +
        landmarks[454].z
    ) / 2

    bridge_projection = abs(
        bridge_z -
        cheek_avg_z
    )

    # -----------------------------------------
    # Orientation
    # -----------------------------------------

    nostril_center_y = (
        left_nostril[1] +
        right_nostril[1]
    ) / 2

    tip_offset = (
        nostril_center_y -
        nose_tip[1]
    )

    # -----------------------------------------
    # Confidence
    # -----------------------------------------

    pose_error = estimate_pose(
        landmarks
    )

    confidence = max(
        0,
        1 - pose_error * 8
    )

    confidence = round(
        confidence,
        2
    )

    result = {}

    result["confidence"] = confidence

    result["width_ratio"] = round(
        width_ratio,
        3
    )

    result["length_ratio"] = round(
        length_ratio,
        3
    )

    result["tip_ratio"] = round(
        tip_ratio,
        3
    )

    result["bridge_projection"] = round(
        bridge_projection,
        4
    )

    # -----------------------------------------
    # Width
    # -----------------------------------------

    if width_ratio > 0.44:

        result["nose_width"] = (
            "Cánh mũi hơi rộng"
        )

    elif width_ratio > 0.35:

        result["nose_width"] = (
            "Cánh mũi cân đối"
        )

    else:

        result["nose_width"] = (
            "Cánh mũi nhỏ"
        )

    # -----------------------------------------
    # Length
    # -----------------------------------------

    if length_ratio > 0.34:

        result["nose_length"] = (
            "Mũi hơi dài"
        )

    elif length_ratio > 0.24:

        result["nose_length"] = (
            "Mũi trung bình"
        )

    else:

        result["nose_length"] = (
            "Mũi hơi ngắn"
        )

    # -----------------------------------------
    # Tip Shape
    # -----------------------------------------

    if tip_ratio > 0.95:

        result["tip_shape"] = (
            "Đầu mũi tròn mềm"
        )

    elif tip_ratio > 0.78:

        result["tip_shape"] = (
            "Đầu mũi cân đối"
        )

    else:

        result["tip_shape"] = (
            "Đầu mũi gọn nhẹ"
        )

    # -----------------------------------------
    # Orientation
    # -----------------------------------------

    if tip_offset > 10:

        result["orientation"] = (
            "Hơi hướng lên nhẹ"
        )

    elif tip_offset < -10:

        result["orientation"] = (
            "Hơi hướng xuống"
        )

    else:

        result["orientation"] = (
            "Neutral / thẳng"
        )

    # -----------------------------------------
    # Bridge
    # -----------------------------------------

    if bridge_projection > 0.13:

        result["bridge"] = (
            "Sống mũi nổi khá rõ"
        )

    elif bridge_projection > 0.07:

        result["bridge"] = (
            "Sống mũi tự nhiên"
        )

    else:

        result["bridge"] = (
            "Sống mũi mềm nhẹ"
        )

    # -----------------------------------------
    # Overall
    # -----------------------------------------

    result["overall"] = (
        "Dáng mũi hài hòa với khuôn mặt"
    )

    return result

# =====================================================
# LOAD IMAGE
# =====================================================

image = cv2.imread(
    IMAGE_PATH
)

if image is None:

    print("Cannot load image")
    exit()

h, w, _ = image.shape

# =====================================================
# RGB
# =====================================================

rgb = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2RGB
)

mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=rgb
)

# =====================================================
# DETECT
# =====================================================

detection_result = detector.detect(
    mp_image
)

if not detection_result.face_landmarks:

    print("No face detected")
    exit()

landmarks = detection_result.face_landmarks[0]

# =====================================================
# NOSE ROI
# =====================================================

nose_indices = [
    1, 2, 6, 98, 327,
    168, 197, 195, 5,
    4
]

nose_points = []

for idx in nose_indices:

    p = get_point(
        landmarks,
        idx,
        w,
        h
    )

    nose_points.append(p)

nose_points = np.array(
    nose_points
)

x, y, nw, nh = cv2.boundingRect(
    nose_points
)

padding = 15

x1 = max(x - padding, 0)
y1 = max(y - padding, 0)

x2 = min(x + nw + padding, w)
y2 = min(y + nh + padding, h)

nose_crop = image[
    y1:y2,
    x1:x2
]

# =====================================================
# PREPROCESS
# =====================================================

gray = cv2.cvtColor(
    nose_crop,
    cv2.COLOR_BGR2GRAY
)

clahe = cv2.createCLAHE(
    clipLimit=1.2,
    tileGridSize=(6, 6)
)

enhanced = clahe.apply(
    gray
)

filtered = cv2.GaussianBlur(
    enhanced,
    (3, 3),
    0
)

kernel = np.array([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
])

sharpened = cv2.filter2D(
    filtered,
    -1,
    kernel
)

edges = cv2.Canny(
    sharpened,
    40,
    100
)

# =====================================================
# CONTOURS
# =====================================================

contours, _ = cv2.findContours(
    edges,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

contour_image = cv2.cvtColor(
    sharpened,
    cv2.COLOR_GRAY2BGR
)

cv2.drawContours(
    contour_image,
    contours,
    -1,
    (0, 255, 0),
    1
)

# =====================================================
# DEBUG IMAGE
# =====================================================

debug_image = image.copy()

cv2.rectangle(
    debug_image,
    (x1, y1),
    (x2, y2),
    (0, 255, 0),
    2
)

for idx in nose_indices:

    p = get_point(
        landmarks,
        idx,
        w,
        h
    )

    cv2.circle(
        debug_image,
        tuple(p),
        3,
        (0, 255, 255),
        -1
    )

# =====================================================
# ANALYSIS
# =====================================================

nose_analysis = analyze_nose(
    landmarks,
    w,
    h
)

(
    facial_analysis,
    hairline,
    brow_line,
    nose_line,
    chin_line
) = analyze_facial_thirds(
    landmarks,
    w,
    h
)

# =====================================================
# DRAW FACIAL THIRDS
# =====================================================

line_points = [
    hairline,
    brow_line,
    nose_line,
    chin_line
]

for p in line_points:

    cv2.line(
        debug_image,
        (0, int(p[1])),
        (w, int(p[1])),
        (255, 0, 0),
        2
    )

# =====================================================
# PRINT RESULTS
# =====================================================

print("\n=================================")
print("ADVANCED FACE ANALYSIS")
print("=================================")

print("\n--- NOSE ANALYSIS ---\n")

for k, v in nose_analysis.items():

    print(f"{k}: {v}")

print("\n--- FACIAL THIRDS ---\n")

for k, v in facial_analysis.items():

    print(f"{k}: {v}")

# =====================================================
# SHOW WINDOWS
# =====================================================

cv2.imshow(
    "Original",
    debug_image
)

cv2.imshow(
    "Nose Crop",
    nose_crop
)

cv2.imshow(
    "Enhanced Nose",
    sharpened
)

cv2.imshow(
    "Edges",
    edges
)

cv2.imshow(
    "Contours",
    contour_image
)

cv2.waitKey(0)

cv2.destroyAllWindows()

I0000 00:00:1778214177.236062 4855355 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 11, prefix = pthread-default
W0000 00:00:1778214177.236493 4855355 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1778214177.279808 4855355 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778214177.280749 4855358 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778214177.285977 4855360 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.



ADVANCED FACE ANALYSIS

--- NOSE ANALYSIS ---

confidence: 0.19
width_ratio: 0.402
length_ratio: 0.269
tip_ratio: 0.816
bridge_projection: 0.3278
nose_width: Cánh mũi cân đối
nose_length: Mũi trung bình
tip_shape: Đầu mũi cân đối
orientation: Hơi hướng lên nhẹ
bridge: Sống mũi nổi khá rõ
overall: Dáng mũi hài hòa với khuôn mặt

--- FACIAL THIRDS ---

upper_ratio: 0.284
middle_ratio: 0.383
lower_ratio: 0.334
balance_score: 0.85
upper_face: Thượng đình cân đối
middle_face: Trung đình hơi dài
lower_face: Hạ đình cân đối
overall_balance: Tỉ lệ gương mặt khá cân đối
